# Akan-BPE Phase 2A1 — Qwen3-0.6B × Akan TTS Tokenizer

Self-contained Colab notebook. Run all cells top-to-bottom.

**Steps:**
1. Clone repo and install deps
2. Download Akan datasets from HuggingFace Hub
3. Run QLoRA fine-tune experiment (`Qwen/Qwen3-0.6B` + Akan TTS tokenizer)
4. Inspect results — fertility reduction, eval loss/perplexity, generation samples

**GPU required.** Before running: Runtime → Change runtime type → T4 GPU.

In [ ]:
# 1. Clone repo (skip if already cloned)
import os

REPO = "https://github.com/attabeezy/akan-bpe.git"
if not os.path.isdir("akan-bpe"):
    !git clone {REPO}
%cd akan-bpe

In [ ]:
# 2. Install dependencies
%pip install -q -e ".[dev,train]"
%pip install -q bitsandbytes sentencepiece

In [ ]:
# 3. Confirm GPU is available
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Go to Runtime → Change runtime type → T4 GPU, then re-run."
    )
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# 4. Download Akan datasets from HuggingFace Hub
# Produces: data/pristine_twi_{train,validation,test}.jsonl
#           data/aka_asr_{train,validation,test}.jsonl
!python scripts/download.py --output-dir data

In [ ]:
# 5. Verify all required inputs exist
from pathlib import Path

required = {
    "TTS train data" : Path("data/pristine_twi_train.jsonl"),
    "TTS test data"  : Path("data/pristine_twi_test.jsonl"),
    "TTS tokenizer"  : Path("models/tts_tokenizer.json"),
}
missing = [name for name, p in required.items() if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing required inputs: {missing}")
print("All inputs ready.")
for name, p in required.items():
    print(f"  {name}: {p}  ({p.stat().st_size / 1e6:.1f} MB)")

In [ ]:
# 6. Run Phase 2A1 experiment
# QLoRA fine-tune on Qwen3-0.6B with the Akan TTS tokenizer.
# Writes model adapters to models/phase2a1_qwen3_0_6b_tts/
# Writes result JSON to results/phase2a1_qwen3_0_6b_tts.json
!python scripts/model_integration.py \
    --experiment-id phase2a1_qwen3_0_6b_tts \
    --model-id Qwen/Qwen3-0.6B \
    --tokenizer-path models/tts_tokenizer.json \
    --train-file data/pristine_twi_train.jsonl \
    --eval-file data/pristine_twi_test.jsonl \
    --output-dir models/phase2a1_qwen3_0_6b_tts \
    --results-output results/phase2a1_qwen3_0_6b_tts.json \
    --device-mode colab-qlora \
    --max-train-samples 4096 \
    --max-eval-samples 512 \
    --max-length 256 \
    --batch-size 1 \
    --grad-accum 16 \
    --epochs 1 \
    --learning-rate 2e-4 \
    --lora-r 16

In [ ]:
# 7. Load result JSON
import json
from pathlib import Path

result = json.loads(
    Path("results/phase2a1_qwen3_0_6b_tts.json").read_text(encoding="utf-8")
)
print("Experiment :", result["experiment_id"])
print("Model      :", result["model_id"])
print("Device     :", result.get("device", {}))

In [ ]:
# 8. Token count comparison — fertility reduction
cmp = result["token_count_comparison"]
print(f"Base tokenizer fertility : {cmp['base']['fertility']:.3f} tokens/word")
print(f"Akan tokenizer fertility : {cmp['experiment']['fertility']:.3f} tokens/word")
print(f"Reduction ratio          : {cmp['reduction_ratio']:.1%}")

In [ ]:
# 9. Eval metrics
ev = result["eval"]
print(f"Eval loss  : {ev['eval_loss']:.4f}")
print(f"Perplexity : {ev['perplexity']:.2f}")

In [ ]:
# 10. Generation samples
for i, s in enumerate(result["generation_samples"], 1):
    print(f"--- Sample {i} ---")
    print(f"Prompt    : {s['prompt']}")
    print(f"Completion: {s['completion']}")
    print()